# 1D-CNN Model Architecture and Training

Model architecture discussion

The model definition is in `form_analyzer.py`

I chose a CNN to do form analysis because the keypoint data has time locality. The motion of each point in time can be used to identify form errors.

Conceptually, the 1D CNN is the same as a 2D CNN for image processing: the model weights in the convolutional layers are the kernels convolved with input data. The layers have multiple output channels, which each have their own learned kernel. The final layer is a linear layer from the final convolutional layer to the four output classes. Each output node corresponds to the model's identification of the four form errors it has learned.

In [6]:
import torch   
device = torch.device("mps")

In [ ]:

import sys
# Add the absolute or relative path to the folder containing your script
sys.path.append('../src')
from src.form_analyzer import create_dataloaders

train_loader, val_loader, test_loader = create_dataloaders("../data/custom_running_form/custom_labels.csv")

Dataset Split -> Train: 553 | Val: 118 | Test: 119


## Training Process

This is the training step, which includes evaluation of the model on the validation set each epoch. Patience is used for early stopping, and dropout is used in both the convolutional layers and final linear layer.

In [ ]:
keypoint_path = "../data/custom_running_form/keypoints"

def train_model(model, optimizer, loss_fn, train_loader, val_loader, num_epochs=20, patience=0, model_save_path="../models/form_analysis/form_analyzer_model.pt"):
    best_val_loss = float('inf')

    patience_count = 0
    prev_val_loss = 1
    
    for epoch in range(num_epochs):
        model.train()
        
        running_train_loss = 0.0
        count_keypoints = 0
        
        for keypoints, labels in train_loader:
            keypoints, labels = keypoints.to(device), labels.to(device)

            optimizer.zero_grad()
            
            outputs = model(keypoints)
            
            loss = loss_fn(outputs, labels)

            loss.backward()
            
            optimizer.step()
            
            running_train_loss += loss.item()
            count_keypoints += 1
            
        epoch_avg_train_loss = running_train_loss / count_keypoints

        model.eval()
        running_val_loss = 0.0
        count_val_keypoints = 0

        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for keypoints, labels in val_loader:
                keypoints, labels = keypoints.to(device), labels.to(device)
                
                outputs = model(keypoints)
                loss = loss_fn(outputs, labels)
                
                running_val_loss += loss.item()
                count_val_keypoints += 1
                
                probs = torch.sigmoid(outputs)
                
                preds = (probs > 0.5).float()
                
                all_preds.append(preds.cpu())
                all_labels.append(labels.cpu())
                
        epoch_avg_val_loss = running_val_loss / count_val_keypoints

        if epoch_avg_val_loss > prev_val_loss:
            patience_count += 1
            if patience_count == patience:
                print("Patience exhausted!")
                break
        else:
            patience_count = 0

        prev_val_loss = epoch_avg_val_loss
        
        # Calculate Exact Match Accuracy 
        all_preds = torch.cat(all_preds)
        all_labels = torch.cat(all_labels)
        exact_match_acc = (all_preds == all_labels).all(dim=1).float().mean().item()
        
        print(f"Epoch [{epoch+1:02d}/{num_epochs:02d}] | Train Loss: {epoch_avg_train_loss:.4f} | Val Loss: {epoch_avg_val_loss:.4f} | Val Exact Match Acc: {exact_match_acc * 100:.2f}%")
        
        if epoch_avg_val_loss < best_val_loss:
            best_val_loss = epoch_avg_val_loss
            print("Lowest val loss achieved, saving model.")
            torch.save(model.state_dict(), model_save_path)
            
    print("Training complete.")

## Kernel Size Comparison

Evaluate the CNN performance on the custom running form dataset with different kernel sizes and patience.

In [ ]:
import torch.nn as nn
import torch.optim as optim
from src.form_analyzer import FormAnalyzer1DCNN

In [ ]:
learning_rate = 0.001

model_3kernel = FormAnalyzer1DCNN(kernel_size=3).to(device)

optimizer = optim.Adam(model_3kernel.parameters(), lr=learning_rate)
loss_fn = nn.BCEWithLogitsLoss()

model_save_path = "../models/form_analysis/form_analyzer_model_3kernel.pt"

train_model(model_3kernel, optimizer, loss_fn, train_loader, val_loader, num_epochs=100, patience=5, model_save_path=model_save_path)

Epoch [01/100] | Train Loss: 0.5896 | Val Loss: 0.5368 | Val Exact Match Acc: 23.73%
Lowest val loss achieved, saving model.
Epoch [02/100] | Train Loss: 0.4699 | Val Loss: 0.4126 | Val Exact Match Acc: 24.58%
Lowest val loss achieved, saving model.
Epoch [03/100] | Train Loss: 0.3473 | Val Loss: 0.2544 | Val Exact Match Acc: 74.58%
Lowest val loss achieved, saving model.
Epoch [04/100] | Train Loss: 0.2435 | Val Loss: 0.2188 | Val Exact Match Acc: 63.56%
Lowest val loss achieved, saving model.
Epoch [05/100] | Train Loss: 0.1907 | Val Loss: 0.1746 | Val Exact Match Acc: 65.25%
Lowest val loss achieved, saving model.
Epoch [06/100] | Train Loss: 0.1770 | Val Loss: 0.2650 | Val Exact Match Acc: 66.95%
Epoch [07/100] | Train Loss: 0.1587 | Val Loss: 0.2845 | Val Exact Match Acc: 61.02%
Epoch [08/100] | Train Loss: 0.1478 | Val Loss: 0.7012 | Val Exact Match Acc: 33.05%
Epoch [09/100] | Train Loss: 0.1333 | Val Loss: 0.1326 | Val Exact Match Acc: 81.36%
Lowest val loss achieved, saving mo

In [ ]:
learning_rate = 0.001

model_5kernel = FormAnalyzer1DCNN(kernel_size=5).to(device)

optimizer = optim.Adam(model_5kernel.parameters(), lr=learning_rate)
loss_fn = nn.BCEWithLogitsLoss()

model_save_path = "../models/form_analysis/form_analyzer_model_5kernel.pt"

train_model(model_5kernel, optimizer, loss_fn, train_loader, val_loader, num_epochs=100, patience=5, model_save_path=model_save_path)

Epoch [01/100] | Train Loss: 0.5642 | Val Loss: 0.5437 | Val Exact Match Acc: 23.73%
Lowest val loss achieved, saving model.
Epoch [02/100] | Train Loss: 0.4180 | Val Loss: 0.3305 | Val Exact Match Acc: 53.39%
Lowest val loss achieved, saving model.
Epoch [03/100] | Train Loss: 0.2693 | Val Loss: 0.3285 | Val Exact Match Acc: 54.24%
Lowest val loss achieved, saving model.
Epoch [04/100] | Train Loss: 0.2229 | Val Loss: 0.5454 | Val Exact Match Acc: 38.14%
Epoch [05/100] | Train Loss: 0.1935 | Val Loss: 0.3090 | Val Exact Match Acc: 53.39%
Lowest val loss achieved, saving model.
Epoch [06/100] | Train Loss: 0.1845 | Val Loss: 0.1917 | Val Exact Match Acc: 77.12%
Lowest val loss achieved, saving model.
Epoch [07/100] | Train Loss: 0.1536 | Val Loss: 0.1741 | Val Exact Match Acc: 72.03%
Lowest val loss achieved, saving model.
Epoch [08/100] | Train Loss: 0.1514 | Val Loss: 0.1095 | Val Exact Match Acc: 83.05%
Lowest val loss achieved, saving model.
Epoch [09/100] | Train Loss: 0.1545 | Va

In [ ]:
learning_rate = 0.001

model_7kernel = FormAnalyzer1DCNN(kernel_size=7).to(device)

optimizer = optim.Adam(model_7kernel.parameters(), lr=learning_rate)

model_save_path = "../models/form_analysis/form_analyzer_model_7kernel.pt"

train_model(model_7kernel, optimizer, loss_fn, train_loader, val_loader, num_epochs=100, patience=5, model_save_path=model_save_path)

Epoch [01/100] | Train Loss: 0.5783 | Val Loss: 0.5330 | Val Exact Match Acc: 23.73%
Lowest val loss achieved, saving model.
Epoch [02/100] | Train Loss: 0.4447 | Val Loss: 0.4430 | Val Exact Match Acc: 27.12%
Lowest val loss achieved, saving model.
Epoch [03/100] | Train Loss: 0.3140 | Val Loss: 0.2924 | Val Exact Match Acc: 56.78%
Lowest val loss achieved, saving model.
Epoch [04/100] | Train Loss: 0.2258 | Val Loss: 0.2369 | Val Exact Match Acc: 56.78%
Lowest val loss achieved, saving model.
Epoch [05/100] | Train Loss: 0.2103 | Val Loss: 0.2603 | Val Exact Match Acc: 57.63%
Epoch [06/100] | Train Loss: 0.1846 | Val Loss: 0.1897 | Val Exact Match Acc: 80.51%
Lowest val loss achieved, saving model.
Epoch [07/100] | Train Loss: 0.1980 | Val Loss: 0.2079 | Val Exact Match Acc: 66.95%
Epoch [08/100] | Train Loss: 0.1618 | Val Loss: 0.2022 | Val Exact Match Acc: 76.27%
Epoch [09/100] | Train Loss: 0.1228 | Val Loss: 0.1114 | Val Exact Match Acc: 77.97%
Lowest val loss achieved, saving mo

## Patience Reduction

Decrease patience from 5 to 3 and evaluate model performance:

In [ ]:
learning_rate = 0.001

model_3kernel_patience3 = FormAnalyzer1DCNN(kernel_size=3).to(device)

optimizer = optim.Adam(model_3kernel_patience3.parameters(), lr=learning_rate)

model_save_path = "../models/form_analysis/form_analyzer_model_3kernel_patience3.pt"

train_model(model_3kernel_patience3, optimizer, loss_fn, train_loader, val_loader, num_epochs=100, patience=3, model_save_path=model_save_path)

Epoch [01/100] | Train Loss: 0.5912 | Val Loss: 0.5440 | Val Exact Match Acc: 23.73%
Lowest val loss achieved, saving model.
Epoch [02/100] | Train Loss: 0.4771 | Val Loss: 0.4604 | Val Exact Match Acc: 24.58%
Lowest val loss achieved, saving model.
Epoch [03/100] | Train Loss: 0.3954 | Val Loss: 0.3274 | Val Exact Match Acc: 60.17%
Lowest val loss achieved, saving model.
Epoch [04/100] | Train Loss: 0.2680 | Val Loss: 0.2430 | Val Exact Match Acc: 55.93%
Lowest val loss achieved, saving model.
Epoch [05/100] | Train Loss: 0.2245 | Val Loss: 0.1904 | Val Exact Match Acc: 81.36%
Lowest val loss achieved, saving model.
Epoch [06/100] | Train Loss: 0.2094 | Val Loss: 0.1567 | Val Exact Match Acc: 77.12%
Lowest val loss achieved, saving model.
Epoch [07/100] | Train Loss: 0.1762 | Val Loss: 0.1349 | Val Exact Match Acc: 79.66%
Lowest val loss achieved, saving model.
Epoch [08/100] | Train Loss: 0.1609 | Val Loss: 0.1093 | Val Exact Match Acc: 89.83%
Lowest val loss achieved, saving model.


In [ ]:
model_5kernel_patience3 = FormAnalyzer1DCNN(kernel_size=5).to(device)

optimizer = optim.Adam(model_5kernel_patience3.parameters(), lr=learning_rate)
loss_fn = nn.BCEWithLogitsLoss()

model_save_path = "../models/form_analysis/form_analyzer_model_5kernel_patience3.pt"

train_model(model_5kernel_patience3, optimizer, loss_fn, train_loader, val_loader, num_epochs=100, patience=3, model_save_path=model_save_path)

Epoch [01/100] | Train Loss: 0.5377 | Val Loss: 0.5308 | Val Exact Match Acc: 23.73%
Lowest val loss achieved, saving model.
Epoch [02/100] | Train Loss: 0.4207 | Val Loss: 0.3403 | Val Exact Match Acc: 44.92%
Lowest val loss achieved, saving model.
Epoch [03/100] | Train Loss: 0.2713 | Val Loss: 0.1910 | Val Exact Match Acc: 76.27%
Lowest val loss achieved, saving model.
Epoch [04/100] | Train Loss: 0.2207 | Val Loss: 0.3032 | Val Exact Match Acc: 47.46%
Epoch [05/100] | Train Loss: 0.1917 | Val Loss: 0.1931 | Val Exact Match Acc: 72.03%
Epoch [06/100] | Train Loss: 0.1866 | Val Loss: 0.3158 | Val Exact Match Acc: 41.53%
Epoch [07/100] | Train Loss: 0.1688 | Val Loss: 0.1893 | Val Exact Match Acc: 73.73%
Lowest val loss achieved, saving model.
Epoch [08/100] | Train Loss: 0.1362 | Val Loss: 0.1529 | Val Exact Match Acc: 80.51%
Lowest val loss achieved, saving model.
Epoch [09/100] | Train Loss: 0.1302 | Val Loss: 0.6951 | Val Exact Match Acc: 37.29%
Epoch [10/100] | Train Loss: 0.1207

In [ ]:
model_7kernel_patience3 = FormAnalyzer1DCNN(kernel_size=7).to(device)

optimizer = optim.Adam(model_7kernel_patience3.parameters(), lr=learning_rate)
loss_fn = nn.BCEWithLogitsLoss()

model_save_path = "../models/form_analysis/form_analyzer_model_7kernel_patience3.pt"

train_model(model_7kernel_patience3, optimizer, loss_fn, train_loader, val_loader, num_epochs=100, patience=3, model_save_path=model_save_path)

Epoch [01/100] | Train Loss: 0.5667 | Val Loss: 0.5333 | Val Exact Match Acc: 25.42%
Lowest val loss achieved, saving model.
Epoch [02/100] | Train Loss: 0.4124 | Val Loss: 0.3332 | Val Exact Match Acc: 35.59%
Lowest val loss achieved, saving model.
Epoch [03/100] | Train Loss: 0.2939 | Val Loss: 0.3192 | Val Exact Match Acc: 42.37%
Lowest val loss achieved, saving model.
Epoch [04/100] | Train Loss: 0.2276 | Val Loss: 0.2104 | Val Exact Match Acc: 73.73%
Lowest val loss achieved, saving model.
Epoch [05/100] | Train Loss: 0.1922 | Val Loss: 0.2097 | Val Exact Match Acc: 60.17%
Lowest val loss achieved, saving model.
Epoch [06/100] | Train Loss: 0.1846 | Val Loss: 0.2341 | Val Exact Match Acc: 63.56%
Epoch [07/100] | Train Loss: 0.1576 | Val Loss: 0.3023 | Val Exact Match Acc: 50.85%
Patience exhausted!
Training complete.


## Hyperparameter Tuning Results

Patience = 5

| Metric | Kernel Size 3   | Kernel Size 5 | Kernel Size 7 |
| ---- | -------- | -------- | ------ |
| Validation Loss   | 0.0007  | 0.0003 | 0.0002 |
| Validation Accuracy  | 100.00%   | 100.00% | 100.00% |
| Training Time (seconds)       | 40.8 | 41.7  | 44.3  |


Patience = 3

| Metric | Kernel Size 3   | Kernel Size 5 | Kernel Size 7 |
| ---- | -------- | -------- | ------ |
| Validation Loss   | 0.0676  | 0.0009 | 0.2097 |
| Validation Accuracy  | 89.83%   | 100.00% | 60.17% |
| Training Time (seconds)       | 7.9 | 40.5  | 3.2  |

Kernel sizes 3 and 7 had significantly early stopping due to patience.

## Testing Models

Evaluate models on the test set. The threshold for identifying an error is raised to 70% probability to avoid false positives.

In [ ]:
keypoint_path = "../data/custom_running_form/keypoints"

def test_model(model, loss_fn, test_loader):
    model.eval()
    running_test_loss = 0.0
    count_features = 0

    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for features, labels in test_loader:
            features, labels = features.to(device), labels.to(device)
            
            outputs = model(features)
            loss = loss_fn(outputs, labels)
            
            running_test_loss += loss.item()
            count_features += 1
            
            probs = torch.sigmoid(outputs)
            
            preds = (probs > 0.7).float()
            
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
            
    avg_test_loss = running_test_loss / count_features
    
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    exact_match_acc = (all_preds == all_labels).all(dim=1).float().mean().item()
        
    print(f"Test Loss: {avg_test_loss:.4f} | Test Exact Match Acc: {exact_match_acc * 100:.2f}%")

In [39]:
loss_fn = nn.BCEWithLogitsLoss()
test_model(model=model_3kernel_patience3, loss_fn=loss_fn, test_loader=test_loader)

Test Loss: 0.2595 | Test Exact Match Acc: 72.27%


In [40]:
test_model(model=model_5kernel_patience3, loss_fn=loss_fn, test_loader=test_loader)

Test Loss: 0.1180 | Test Exact Match Acc: 85.71%


In [41]:
test_model(model=model_7kernel_patience3, loss_fn=loss_fn, test_loader=test_loader)

Test Loss: 0.5007 | Test Exact Match Acc: 47.06%


In [42]:
test_model(model=model_3kernel, loss_fn=loss_fn, test_loader=test_loader)

Test Loss: 0.0718 | Test Exact Match Acc: 86.55%


In [43]:
test_model(model=model_5kernel, loss_fn=loss_fn, test_loader=test_loader)

Test Loss: 0.2007 | Test Exact Match Acc: 81.51%


In [44]:
test_model(model=model_7kernel, loss_fn=loss_fn, test_loader=test_loader)

Test Loss: 0.0632 | Test Exact Match Acc: 99.16%


## Test Performance

| Metric | Kernel Size 3, Patience 3   | Kernel Size 5, Patience 3 | Kernel Size 7, Patience 3 | Kernel Size 3, Patience 5 | Kernel Size 5, Patience 5 | Kernel Size 7, Patience 5 |
| ---- | -------- | -------- | ------ | -------- | -------- | ------ |
| Test Loss   | 0.2595 | 0.1180 | 0.5007 | 0.0718  | 0.2007 | 0.0632 |
| Test Accuracy  | 72.27% | 85.71% | 47.06% | 86.55%  | 81.51% | 99.16% |


This informed my decision to use the 1D CNN model with kernel size 7 and patience 5 to evaluate user submissions.